In [ ]:

# BRAIN TUMOR MRI IMAGE CLASSIFICATION
# Google Colab - Complete Training and Evaluation Code

import os
import zipfile
import shutil
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from google.colab import files
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# 1. SETTINGS
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10
SEED = 42

tf.keras.utils.set_random_seed(SEED)

# 2. UPLOAD DATASET ZIP
print("Upload your MRI dataset ZIP file.")
uploaded = files.upload()

zip_files = [name for name in uploaded if name.lower().endswith(".zip")]
if not zip_files:
    raise ValueError("Please upload a ZIP file containing the MRI dataset.")

ZIP_PATH = zip_files[0]
EXTRACT_DIR = "/content/brain_tumor_dataset"

if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    # Avoid extracting unsafe paths outside the target directory.
    root = os.path.realpath(EXTRACT_DIR)
    for member in z.infolist():
        destination = os.path.realpath(
            os.path.join(EXTRACT_DIR, member.filename)
        )
        if destination != root and not destination.startswith(root + os.sep):
            raise ValueError("Unsafe path found inside the ZIP file.")
    z.extractall(EXTRACT_DIR)

print("Dataset extracted successfully.")


def find_folder(base, folder_name):
    for current, dirs, _ in os.walk(base):
        dirs[:] = [d for d in dirs if d != "__MACOSX"]
        if os.path.basename(current).lower() == folder_name.lower():
            return current
    return None

train_dir = find_folder(EXTRACT_DIR, "Training")
test_dir = find_folder(EXTRACT_DIR, "Testing")

if train_dir is None:
    raise ValueError(
        "Training folder not found. ZIP should contain "
        "Training/class_name/image.jpg folders."
    )

if test_dir is None:
    print("Testing folder not found. Creating a validation split "
          "from the Training folder; final test metrics will be unavailable.")
    test_dir = None


train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=10,
    zoom_range=0.10,
    horizontal_flip=True,
    validation_split=0.20 if test_dir is None else 0.0
)

eval_datagen = ImageDataGenerator(rescale=1.0 / 255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training" if test_dir is None else None,
    seed=SEED,
    shuffle=True
)

if test_dir is None:
    val_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        subset="validation",
        seed=SEED,
        shuffle=False
    )
    test_generator = None
else:
    val_generator = eval_datagen.flow_from_directory(
        train_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        seed=SEED,
        shuffle=False
    )
    test_generator = eval_datagen.flow_from_directory(
        test_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        shuffle=False
    )

class_names = list(train_generator.class_indices.keys())
num_classes = len(class_names)

if num_classes < 2:
    raise ValueError("At least two class folders with images are required.")

if test_generator is not None:
    if set(test_generator.class_indices) != set(train_generator.class_indices):
        raise ValueError("Training and Testing must contain the same class folders.")

print("Classes:", class_names)
print("Class indices:", train_generator.class_indices)

# 5. BUILD CNN MODEL
model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),

    layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(256, (3, 3), activation="relu", padding="same"),
    layers.MaxPooling2D((2, 2)),

    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# 6. TRAIN MODEL
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    )
]

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks
)

# 7. PLOT ACCURACY AND LOSS
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()


if test_generator is not None:
    test_generator.reset()
    loss, accuracy = model.evaluate(test_generator, verbose=1)
    print(f"Test Loss: {loss:.4f}")
    print(f"Test Accuracy: {accuracy * 100:.2f}%")

    predictions = model.predict(test_generator, verbose=1)
    y_pred = np.argmax(predictions, axis=1)
    y_true = test_generator.classes

    print("\nClassification Report:")
    print(classification_report(
        y_true,
        y_pred,
        labels=list(range(num_classes)),
        target_names=class_names,
        zero_division=0
    ))

    cm = confusion_matrix(
        y_true, y_pred, labels=list(range(num_classes))
    )
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )
    plt.xlabel("Predicted Class")
    plt.ylabel("Actual Class")
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.show()
else:
    print("No separate test set was available.")
    print("Use an independent test dataset for final evaluation.")

MODEL_PATH = "/content/brain_tumor_cnn.keras"
model.save(MODEL_PATH)
print("Model saved:", MODEL_PATH)


files.download(MODEL_PATH)


print("\nUpload one MRI image to test the trained model.")
new_upload = files.upload()

image_extensions = (".jpg", ".jpeg", ".png", ".bmp")
image_files = [
    name for name in new_upload
    if name.lower().endswith(image_extensions)
]

if image_files:
    image_path = image_files[0]

    img = tf.keras.utils.load_img(
        image_path, target_size=IMG_SIZE
    )
    img_array = tf.keras.utils.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    probabilities = model.predict(img_array, verbose=0)[0]
    predicted_index = int(np.argmax(probabilities))

    print("Predicted class:", class_names[predicted_index])
    print(f"Model confidence: {probabilities[predicted_index] * 100:.2f}%")

    plt.imshow(img)
    plt.title(
        f"Prediction: {class_names[predicted_index]}"
    )
    plt.axis("off")
    plt.show()
else:
    print("No supported MRI image uploaded for prediction.")


Upload your MRI dataset ZIP file.


Saving archive.zip to archive.zip
Dataset extracted successfully.
Found 5600 images belonging to 4 classes.
Found 5600 images belonging to 4 classes.
Found 1600 images belonging to 4 classes.
Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Class indices: {'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 28, 28, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 421,828 (1.61 MB)

 Trainable params: 421,828 (1.61 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 1160s 7s/step - accuracy: 0.4968 - loss: 1.0739 - val_accuracy: 0.4593 - val_loss: 1.1662
Epoch 2/10
175/175 ━━━━━━━━━━━━━━━━━━━━ 1155s 7s/step - accuracy: 0.6114 - loss: 0.8997 - val_accuracy: 0.4575 - val_loss: 1.2928
Epoch 3/10
171/175 ━━━━━━━━━━━━━━━━━━━━ 20s 5s/step - accuracy: 0.6501 - loss: 0.8354